## Section 12 — One-Class SVM
**Spacecraft Telemetry Anomaly Detection — Phase 2**

- Learns a tight hypersphere boundary around normal data in high-dimensional space
- Any point falling outside the boundary is flagged as anomalous
- Trained unsupervised on clean normal rows only

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings, time, json
warnings.filterwarnings('ignore')
np.random.seed(42)

from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, roc_curve,
    f1_score, precision_score, recall_score
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.facecolor':'white', 'axes.facecolor':'#f8f9fa'})
os.makedirs('plots_v2', exist_ok=True)
os.makedirs('models', exist_ok=True)
print('Libraries loaded.')

### 12.1 Load & Prepare Data

In [ ]:
# Same pipeline as Section 11 — identical data prep for fair comparison
tel = pd.read_csv('data/telemetry_with_anomalies.csv', parse_dates=['timestamp'])
tel = tel.sort_values(['parameter','timestamp']).reset_index(drop=True)

tel_wide = tel.pivot_table(index='timestamp', columns='parameter',
                            values='value', aggfunc='mean').reset_index()
tel_wide.columns.name = None
tel_wide = tel_wide.sort_values('timestamp').reset_index(drop=True)

label_wide = tel.pivot_table(index='timestamp', columns='parameter',
                              values='is_anomaly', aggfunc='max').reset_index()
label_wide.columns.name = None
label_wide = label_wide.sort_values('timestamp').reset_index(drop=True)

param_cols = [c for c in tel_wide.columns if c != 'timestamp']
tel_wide[param_cols]   = tel_wide[param_cols].ffill().bfill()
label_wide[param_cols] = label_wide[param_cols].ffill().fillna(0)

y_true = label_wide[param_cols].max(axis=1).astype(int).values
X_all  = tel_wide[param_cols].values

print(f'Dataset loaded: {tel.shape}')
print(f'Wide-format shape: {X_all.shape}')
print(f'Anomaly rows (wide, with ffill): {y_true.sum()} / {len(y_true)}')

### 12.2 Train / Test Split & Scaling

In [ ]:
# Same 70% split as Section 11 for fair comparison
cutoff     = int(0.70 * len(X_all))
train_mask = (y_true[:cutoff] == 0)
X_train    = X_all[:cutoff][train_mask]

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_all_scaled   = scaler.transform(X_all)

print(f'Training rows (normal only, first 70%): {len(X_train_scaled):,}')
print(f'Scoring rows  (full dataset)          : {len(X_all_scaled):,}')

# Row-level anomaly type label
type_wide = tel.pivot_table(index='timestamp', columns='parameter',
                             values='anomaly_type', aggfunc='first').reset_index()
type_wide.columns.name = None
type_wide = type_wide.sort_values('timestamp').reset_index(drop=True)
def row_type(row):
    for v in row.values:
        if pd.notna(v) and v != 'none': return v
    return 'none'
type_col = type_wide.drop(columns='timestamp').apply(row_type, axis=1)

### 12.3 Train One-Class SVM

In [ ]:
# kernel='rbf' : Radial Basis Function — maps data to high-dimensional space
#                where a smooth boundary is easier to draw
# nu=0.03      : upper bound on fraction of training errors AND support vector fraction
#                set to match our ~3% injection rate
# gamma='scale': auto-computed as 1 / (n_features * X.var())
t0 = time.time()
ocsvm = OneClassSVM(kernel='rbf', nu=0.03, gamma='scale')
ocsvm.fit(X_train_scaled)
t_train = time.time() - t0

print(f'Model trained in {t_train:.2f}s')
print(f'Support vectors: {ocsvm.support_vectors_.shape[0]}')
print('  (these are the boundary-defining training points)')
print()
print('How it works: draws the tightest sphere enclosing normal data.')
print('Points outside = anomalies. RBF kernel makes the boundary non-linear.')

### 12.4 Score & Predict

In [ ]:
t0 = time.time()
scores_svm = ocsvm.decision_function(X_all_scaled)  # signed distance from boundary
                                                      # negative = outside = anomaly
y_pred_svm = ocsvm.predict(X_all_scaled)             # +1 = inside (normal), -1 = outside (anomaly)
t_score    = time.time() - t0

y_pred_svm_bin = (y_pred_svm == -1).astype(int)
scores_svm_inv = -scores_svm   # flip: higher = more anomalous

prec_svm = precision_score(y_true, y_pred_svm_bin, zero_division=0)
rec_svm  = recall_score(y_true, y_pred_svm_bin, zero_division=0)
f1_svm   = f1_score(y_true, y_pred_svm_bin, zero_division=0)
auc_svm  = roc_auc_score(y_true, scores_svm_inv)
cm_svm   = confusion_matrix(y_true, y_pred_svm_bin)
tn_s, fp_s, fn_s, tp_s = cm_svm.ravel()
fpr_svm, tpr_svm, _ = roc_curve(y_true, scores_svm_inv)

print('ONE-CLASS SVM RESULTS')
print(f'  Precision : {prec_svm:.4f}')
print(f'  Recall    : {rec_svm:.4f}')
print(f'  F1 Score  : {f1_svm:.4f}')
print(f'  AUC-ROC   : {auc_svm:.4f}  (primary metric — threshold-independent)')
print(f'  TP={tp_s}  FP={fp_s}  FN={fn_s}  TN={tn_s}')
print(f'  Train time: {t_train:.2f}s | Score time: {t_score:.3f}s')

### 12.5 PCA Visualisation + Confusion Matrix

In [ ]:
# Compress 50 dimensions to 2 for visualisation (approximation)
pca  = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_all_scaled)
var_e = pca.explained_variance_ratio_.sum() * 100

m_nn = (y_pred_svm_bin==0) & (y_true==0)   # true negatives
m_tp = (y_pred_svm_bin==1) & (y_true==1)   # caught anomalies
m_fn = (y_pred_svm_bin==0) & (y_true==1)   # missed anomalies
m_fp = (y_pred_svm_bin==1) & (y_true==0)   # false alarms

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('One-Class SVM — Evaluation', fontsize=12, fontweight='bold')

# PCA scatter
axes[0].scatter(X_2d[m_nn,0], X_2d[m_nn,1], c='#1f77b4', s=6,  alpha=0.3, label='TN (normal, correct)')
axes[0].scatter(X_2d[m_tp,0], X_2d[m_tp,1], c='#d62728', s=50, alpha=0.9, label='TP (anomaly caught)', marker='*')
axes[0].scatter(X_2d[m_fn,0], X_2d[m_fn,1], c='#ff7f0e', s=40, alpha=0.9, label='FN (anomaly MISSED)', marker='v')
axes[0].scatter(X_2d[m_fp,0], X_2d[m_fp,1], c='#bcbd22', s=20, alpha=0.7, label='FP (false alarm)',   marker='x')
axes[0].set_title(f'PCA 2D Projection ({var_e:.1f}% variance explained)')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].legend(fontsize=7)

# Confusion matrix
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Pred Normal','Pred Anomaly'],
            yticklabels=['True Normal','True Anomaly'],
            ax=axes[1], cbar=False, annot_kws={'size':12, 'weight':'bold'})
axes[1].set_title('Confusion Matrix')

plt.tight_layout()
plt.savefig('plots_v2/12_ocsvm_eval.png', dpi=150, bbox_inches='tight')
plt.show()

### 12.6 Per Anomaly-Type Recall

In [ ]:
type_results_svm = []
for atype in ['point', 'contextual', 'collective']:
    mask = type_col == atype
    if mask.sum() > 0:
        yt = (type_col[mask] != 'none').astype(int).values
        yp = y_pred_svm_bin[mask.values]
        type_results_svm.append({
            'Anomaly Type': atype,
            'Total Rows'  : int(mask.sum()),
            'Detected'    : int((yp == 1).sum()),
            'Recall'      : round(recall_score(yt, yp, zero_division=0), 3),
            'Precision'   : round(precision_score(yt, yp, zero_division=0), 3)
        })
display(pd.DataFrame(type_results_svm))

# Save results
svm_results = {
    'model'      : 'One-Class SVM',
    'precision'  : round(prec_svm, 4),
    'recall'     : round(rec_svm, 4),
    'f1'         : round(f1_svm, 4),
    'auc'        : round(auc_svm, 4),
    'train_time' : round(t_train, 3),
    'score_time' : round(t_score, 3),
    'tp': int(tp_s), 'fp': int(fp_s), 'fn': int(fn_s), 'tn': int(tn_s),
    'by_type': type_results_svm,
    'fpr': fpr_svm.tolist(),
    'tpr': tpr_svm.tolist()
}
with open('models/ocsvm_results.json', 'w') as f:
    json.dump(svm_results, f, indent=2)
print('Saved: models/ocsvm_results.json')